In [1]:
using ComputableDAGs
using Pkg
Pkg.develop(; path="/home/reinha57/repos/QEDFeynman.jl/")
using QEDFeynman
using RuntimeGeneratedFunctions
using BenchmarkTools
using QEDcore, QEDprocesses
using Logging
using JLD2
using CUDA

using QEDbase.Mocks

RuntimeGeneratedFunctions.init(@__MODULE__)

MODEL = PerturbativeQED()

┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ ComputableDAGs
│  └─ QEDFeynman
└ @ Base.Precompilation precompilation.jl:650
   Resolving package versions...
  No Changes to `~/repos/ComputableDAGs.jl/Project.toml`
  No Changes to `~/repos/ComputableDAGs.jl/Manifest.toml`
┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ ComputableDAGs
│  └─ QEDFeynman
└ @ Base.Precompilation precompilation.jl:650
┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ ComputableDAGs
│  └─ QEDFeynman
└ @ Base.Precompilation precompilation.jl:650


perturbative QED

In [2]:
N = 56 * 256
INSTANCE_STR = "ke->ke"
INSTANCE = parse_process(INSTANCE_STR, QEDModel())
g = graph(INSTANCE)

inputs = [
    MockPhaseSpacePoint(
        INSTANCE,
        MODEL,
        FlatPhaseSpaceLayout(TwoBodyRestSystem()),
        tuple((rand(MockMomentum) for _ in 1:number_incoming_particles(INSTANCE))...),
        tuple((rand(MockMomentum) for _ in 1:number_outgoing_particles(INSTANCE))...),
    ) for _ in 1:N
]
cu_inputs = CUDA.cu(inputs)
cu_outputs = CUDA.cu([0.0 for _ in 1:N])
k_unopt = eval(kernel(CUDAGPU, g, INSTANCE, @__MODULE__))

f = get_compute_function(g, INSTANCE, cpu_st(), @__MODULE__; closures_size=0, concrete_input_type=typeof(inputs[1]))

ErrorException: Tuple field type cannot be Union{}

In [4]:
optimize_to_fixpoint!(ReductionOptimizer(), g)
k_opt = eval(kernel(CUDAGPU, g, INSTANCE, @__MODULE__))

compute__bc60d522_1539_11f0_0ee0_4d70237f42bf (generic function with 1 method)

In [5]:
K = @cuda launch = false k_unopt(cu_inputs, cu_outputs, N)
K_opt = @cuda launch = false k_opt(cu_inputs, cu_outputs, N)

GPUCompiler.InvalidIRError: InvalidIRError: compiling MethodInstance for compute__b65f3ccc_1539_11f0_1ad5_733445fe785e(::CuDeviceVector{MockInPhaseSpacePoint{ScatteringProcess{Tuple{Photon, Electron}, Tuple{Photon, Electron}, Tuple{PolarizationX, SpinUp}, Tuple{PolarizationX, SpinUp}}, PerturbativeQED, FlatPhaseSpaceLayout{TwoBodyTargetSystem{Energy{2}}}, Tuple{ParticleStateful{Incoming, Photon, MockMomentum{Float64}}, ParticleStateful{Incoming, Electron, MockMomentum{Float64}}}, Tuple{ParticleStateful{Outgoing, Photon, MockMomentum{Float64}}, ParticleStateful{Outgoing, Electron, MockMomentum{Float64}}}}, 1}, ::CuDeviceVector{Float64, 1}, ::Int64) resulted in invalid LLVM IR
Reason: unsupported dynamic function invocation (call to ParticleStateful{Outgoing, Photon, MockMomentum})
Stacktrace:
 [1] #20
   @ ./none:12
 [2] compute__b65f3ccc_1539_11f0_1ad5_733445fe785e
   @ ./none:8
Hint: catch this exception as `err` and call `code_typed(err; interactive = true)` to introspect the erronou

In [ ]:
K_inlined = @cuda launch = false always_inline = true k_unopt(cu_inputs, cu_outputs, N)

In [ ]:
@info CUDA.memory(K)
@device_code_ptx io = open("unopt.ptx", write=true) @cuda launch = false k_unopt(cu_inputs, cu_outputs, N)
CUDA.@device_code dir = "./devcode_unopt" @cuda launch = false k_unopt(cu_inputs, cu_outputs, N)
@info CUDA.memory(K_opt)
@device_code_ptx io = open("opt.ptx", write=true) @cuda launch = false k_opt(cu_inputs, cu_outputs, N)
CUDA.@device_code dir = "./devcode_opt" @cuda launch = false k_opt(cu_inputs, cu_outputs, N)
@info CUDA.memory(K_inlined)
@device_code_ptx io = open("inlined.ptx", write=true) @cuda launch = false always_inline = true k_opt(cu_inputs, cu_outputs, N)

@info CUDA.registers(K)
@info CUDA.registers(K_opt)
@info CUDA.registers(K_inlined)

In [ ]:
CUDA.@sync (@cuda threads=32 blocks=N÷32 k_opt(cu_inputs, cu_outputs, N))

In [ ]:
CUDA.@sync (@cuda threads = 32 blocks = N ÷ 32 k_unopt(cu_inputs, cu_outputs, N))

In [ ]:
g

In [ ]:
CUDA.@profile (CUDA.@sync (@cuda threads=32 blocks=N÷32 k_opt(cu_inputs, cu_outputs, N)))
CUDA.@profile (CUDA.@sync (@cuda threads=32 blocks=N÷32 k_unopt(cu_inputs, cu_outputs, N)))